# Figure 2 panel-generation utilities

This notebook reads `data/testing_2.xlsx` and generates the plotted validation/prediction panels associated with Fig. 2(i–o). Launch Jupyter from the repository root so that relative paths resolve correctly. Code for Fig. 2(j–k) that was originally generated in Google Colab is not included.


In [ ]:
import matplotlib as mpl
mpl.rcParams['xtick.major.size'] = 8
mpl.rcParams['ytick.major.size'] = 8
mpl.rcParams['xtick.major.width'] = 3
mpl.rcParams['ytick.major.width'] = 3

In [ ]:
# Use serif + bold global settings before drawing
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ===== Read data =====
data_path = "data/testing_2.xlsx"
df = pd.read_excel(data_path)

# ===== Global font style =====
plt.rcParams.update({
    "font.size": 20,
    "font.weight": "bold",
    "font.family": "serif"
})

def beautify_ax(ax):
    """Beautify axes: thicken spines and ticks"""
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
    ax.tick_params(axis='both', which='major', width=3)

# ==============
# 1) EpH vs EpH_calc
# ==============
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df["EpH_calc"], df["EpH"], alpha=0.8, s=70, label="Samples")
lo, hi = df["EpH_calc"].min(), df["EpH_calc"].max()
ax.plot([lo, hi], [lo, hi], 'r--', lw=2.5)
ax.set_xlabel("$E_H^{\prime}$ (Calc) (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=10)
ax.set_ylabel("$E_H^{\prime}$ (Pred) (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=22)
ax.legend(prop={'size':20, 'weight':'bold', 'family':'serif'})
beautify_ax(ax)
plt.tight_layout()
plt.show()

# ==============
# 2) EppH vs EppH_calc
# ==============
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df["EppH_calc"], df["EppH"], alpha=0.8, s=70,  label="Samples")
lo, hi = df["EppH_calc"].min(), df["EppH_calc"].max()
ax.plot([lo, hi], [lo, hi], 'r--', lw=2.5)
ax.set_xlabel("$E_H^{\prime\prime}$ (Calc) (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=10)
ax.set_ylabel("$E_H^{\prime\prime}$ (Pred) (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=22)
ax.legend(prop={'size':20, 'weight':'bold', 'family':'serif'})

from matplotlib.ticker import MultipleLocator
ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(base=1))
ax.yaxis.set_major_locator(MultipleLocator(base=1))

beautify_ax(ax)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# —— 确保是数值并去掉 NaN ——
for col in ["EpH_relative_error", "EppH_relative_error"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
dfPlot = df.dropna(subset=["EpH_relative_error", "EppH_relative_error"])

# —— 提取数据 ——
y1 = dfPlot["EpH_relative_error"].values
y2 = dfPlot["EppH_relative_error"].values

fig, ax = plt.subplots(figsize=(8, 8))

# —— 统一 y 轴的箱线图：两组一起传入 —— 
bp = ax.boxplot(
    [y1, y2],
    positions=[1, 2],
    widths=0.4,
    patch_artist=True,
    showfliers=False,        # 若想隐藏离群点：False
    whis=1.5,               # 可改为 (5,95) 用分位数当 whiskers
    medianprops=dict(color="red", linewidth=3),
    boxprops=dict(linewidth=2.5),
    whiskerprops=dict(linewidth=2.5),
    capprops=dict(linewidth=2.5),
    flierprops=dict(
        marker="o",
        markersize=6,
        markerfacecolor="none",  
        markeredgecolor="black",
        linewidth=1.5
    ),
)

# —— 填充箱体颜色（和你之前风格一致：蓝/绿） ——
box_colors = ["skyblue", "lightgreen"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)

# —— x 轴刻度与标签 —— 
ax.set_xticks([1, 2])
ax.set_xlim(0.5, 2.5)
ax.set_xticklabels([r"$E_H^{\prime}$", r"$E_H^{\prime\prime}$"], fontsize=22, fontweight="bold")

# —— y 轴：自动根据两组决定上限；若想固定也可手动改 —— 
ymax = float(np.nanmax(np.concatenate([np.abs(y1), np.abs(y2)])))
ax.set_ylim(ax.set_ylim(0, 5))  # 两组同一坐标轴
ax.set_ylabel(r"Abs. percent error (%)", fontsize=22, fontweight="bold", labelpad=10)

# —— 加粗坐标轴与刻度 —— 
for spine in ax.spines.values():
    spine.set_linewidth(3)
ax.tick_params(axis="both", which="major", labelsize=20, width=3, length=8)
ax.tick_params(axis='x', pad=15)

# —— 在箱线图顶部添加数值标签（显示中位数，放在朝上的error bar上部） ——
medians = [np.median(y1), np.median(y2)]
positions = [1, 2]
# 计算每个箱线图的上whisker顶部位置（朝上的error bar顶部）
# 上whisker顶部 = Q3 + 1.5*IQR（与boxplot的whis=1.5一致）
def get_upper_whisker_top(data):
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    upper_whisker = q3 + 1.07 * iqr
    # 如果计算值超过最大值，则取最大值（因为showfliers=False）
    return min(upper_whisker, np.max(data))
upper_whisker_tops = [get_upper_whisker_top(y1), get_upper_whisker_top(y2)]
for pos, median_val, whisker_top in zip(positions, medians, upper_whisker_tops):
    # 将标签放在朝上的error bar顶部稍上方
    ax.text(pos, whisker_top * 1.02, f'{median_val:.4f}', 
            ha='center', va='bottom', fontsize=18, fontweight='bold', fontfamily='serif')  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# —— 确保为数值并去掉 NaN ——
for col in ["EpH_error", "EppH_error"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df_abs = df.dropna(subset=["EpH_error", "EppH_error"])

y1 = df_abs["EpH_error"].values      # %
y2 = df_abs["EppH_error"].values     # %

# —— 全局字体与风格（与前一张一致） ——
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.default"] = "it"

fig, ax = plt.subplots(figsize=(8, 8))

bp = ax.boxplot(
    [y1, y2],
    positions=[1, 2],
    widths=0.4,
    patch_artist=True,
    showfliers=False,        # 若不想显示离群点：False
    whis=1.5,               # 可改为 (5,95) 用分位数画须
    medianprops=dict(color="red", linewidth=3),
    boxprops=dict(linewidth=2.5),
    whiskerprops=dict(linewidth=2.5),
    capprops=dict(linewidth=2.5),
    flierprops=dict(
        marker="o",
        markersize=6,
        markerfacecolor="none",     
        markeredgecolor="black",
        linewidth=1.5
    ),
)

# —— 箱体填充色（与上一张相同） ——
for patch, color in zip(bp["boxes"], ["skyblue", "lightgreen"]):
    patch.set_facecolor(color)

# —— 轴与刻度 —— 
ax.set_xticks([1, 2])
ax.set_xlim(0.5, 2.5)
ax.set_xticklabels([r"$E_H^{\prime}$", r"$E_H^{\prime\prime}$"], fontsize=22, fontweight="bold")

# 如果更想自适应，请改用：
ymax = float(np.nanmax(np.concatenate([np.abs(y1), np.abs(y2)])))
ax.set_ylim(0, max(0.2, 1.15 * ymax))
ax.set_ylabel("Abs. error (MPa)", fontsize=22, fontweight="bold", labelpad=10)

from matplotlib.ticker import MultipleLocator
ax.yaxis.set_major_locator(MultipleLocator(base=0.05)) 

# —— 加粗坐标轴与刻度 —— 
for spine in ax.spines.values():
    spine.set_linewidth(3)
ax.tick_params(axis="both", which="major", labelsize=20, width=3, length=8)
ax.tick_params(axis='x', pad=15)

# —— 在箱线图顶部添加数值标签（显示中位数，放在朝上的error bar上部） ——
medians = [np.median(y1), np.median(y2)]
positions = [1, 2]
# 计算每个箱线图的上whisker顶部位置（朝上的error bar顶部）
# 上whisker顶部 = Q3 + 1.5*IQR（与boxplot的whis=1.5一致）
def get_upper_whisker_top(data):
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    upper_whisker = q3 + 1.3 * iqr
    # 如果计算值超过最大值，则取最大值（因为showfliers=False）
    return min(upper_whisker, np.max(data))
upper_whisker_tops = [get_upper_whisker_top(y1), get_upper_whisker_top(y2)]
for pos, median_val, whisker_top in zip(positions, medians, upper_whisker_tops):
    # 将标签放在朝上的error bar顶部稍上方
    ax.text(pos, whisker_top * 1.02, f'{median_val:.4f}', 
            ha='center', va='bottom', fontsize=18, fontweight='bold', fontfamily='serif')  

In [ ]:
# ==============
# 3) EpH vs EppH scatter plot
# ==============
fig, ax = plt.subplots(figsize=(20, 10))

# 绘制散点图：EpH为x轴，EppH为y轴
ax.scatter(df["EpH"], df["EppH"], alpha=0.7, s=200)

# 设置坐标轴标签
ax.set_xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
ax.set_ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)

plt.xticks(fontsize=24, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=24, fontweight='bold', fontfamily='serif')

# 美化坐标轴（不设置equal aspect）
for spine in ax.spines.values():
    spine.set_linewidth(3)
ax.tick_params(axis='both', which='major', length=8, width=3)

from matplotlib.ticker import MultipleLocator
ax = plt.gca()
ax.yaxis.set_major_locator(MultipleLocator(base=1))

plt.tight_layout()
plt.show()